In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from spreg import ML_Error, ML_Lag
from libpysal.weights import lag_spatial

# -----------------------------
# 设置路径
# -----------------------------
grid_folder = r"E:\seoul\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# 准备变量
# -----------------------------
results_list = []
def show_formula(model, model_type='SDM'):
    # 处理 y 名称
    y_name = model.name_y if isinstance(model.name_y, str) else model.name_y[0]

    coefs = model.betas.flatten()
    vars_ = model.name_x

    terms = []
    for coef, var in zip(coefs, vars_):
        if var.lower() in ['const', 'constant']:  # 常数项
            terms.append(f"{coef:.4f}")
        else:
            terms.append(f"{coef:.4f}*{var}")

    formula = f"{y_name} = "

    # SDEM lambda 处理
    if model_type == 'SDEM' and hasattr(model, 'lambda_'):
        terms.append(f"{model.lambda_:.4f}*error")

    formula += " + ".join(terms)
    return formula


param_list = []

for year in [2016]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
    explanatory_vars_clean = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR']

    for filename in os.listdir(grid_folder):
        if filename.endswith('.shp') and filename.startswith(f'city{year}_lst_ratio_grid_120m'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            gdf = gdf.replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                # 只保留完整数据
                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                data_gdf = gdf.loc[data.index]
                print(f"CRS: {data_gdf.crs}, File: {filename}, Target: {target}")

                # -----------------------------
                # 空间权重矩阵
                # -----------------------------
                threshold = 1000
                w = libpysal.weights.DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                w.transform = 'r'  # 行标准化
                w_name = 'W1000'
                ds_name = f'yr{year}'

                # -----------------------------
                # y 与 X
                # -----------------------------
                yi = data[target].values.reshape(-1, 1)
                X_main = data[explanatory_vars].values
                WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
                X_all = np.hstack([np.ones((len(data), 1)), X_main, WX_clean])
                name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars_clean]
                # print(len(name_x), X_all.shape[1])
                
                n, k = X_main.shape  # 原始自变量数量

                # -----------------------------
                # 模型估计
                # -----------------------------
                models = {}

                models['SDEM'] = ML_Error(
                    yi, X_all, w=w,
                    name_y=target, name_x=name_x,
                    name_w=w_name, name_ds=ds_name,
                    spat_diag=True,
                    method='LU'   # ←←← 加这行
                )

                # -----------------------------
                # 提取 AIC/BIC/logLik
                # -----------------------------
                for model_name, model in models.items():
                    loglik = model.logll
                    aic = model.aic

                    # 参数数量估算
                    if model_name in ['SDM', 'SDEM']:
                        n_params = 2 * k + 3  # k原始 + k滞后 + ρ/λ + 常数 + σ²
                    else:
                        n_params = k + 2

                    bic = -2 * loglik + n_params * np.log(n)

                    results_list.append({
                        "Year": year,
                        "Grid": filename.split("_")[4],
                        "Target": target,
                        "Model": model_name,
                        "AIC": round(aic, 2),
                        "BIC": round(bic, 2),
                        "LogLik": round(loglik, 2),
                        "N": n
                    })
                    formula_sdem = show_formula(models['SDEM'], model_type='SDEM')

                    # 系数转成 dict，列名是变量名
                    coef_sdem = dict(zip(models['SDEM'].name_x, models['SDEM'].betas.flatten()))

                    #  lambda_ 加进去, rho 不用加是因为已经有Wy了
                    if hasattr(models['SDEM'], 'lambda_'):
                        coef_sdem['lambda'] = models['SDEM'].lambda_

                param_list.append({
                    "Year": year,
                    "Grid": filename.split("_")[4],
                    "Target": target,
                    "Model": "SDEM",
                    "Formula": formula_sdem,
                    **coef_sdem
                })


# -----------------------------
# 汇总输出
# -----------------------------
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(["Year", "Target", "Model"])
output_path = os.path.join(output_folder, "SDM_SDEM_clean_AIC_BIC_120m_2016.xlsx")
results_df.to_excel(output_path, index=False)
all_params_df = pd.DataFrame(param_list)
param_output = os.path.join(output_folder, "SDEM_all_params_120m_2016.xlsx")
all_params_df.to_excel(param_output, index=False)


print(f"✅ 所有模型（SDM/SDEM）AIC/BIC 结果已保存到：\n{output_path}")

c:\Users\JUJUBE\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


CRS: EPSG:5181, File: city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: nor_2016


c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\sparse\_data.py:117: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\JUJUBE\anaconda3\lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\JUJUBE\anaconda3\lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  W.__init__(
c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  warn("Method 'bounded' does not support relative tolerance in x; "
c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\sparse\_index.py:146: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more ef

MemoryError: Unable to allocate 6.03 GiB for an array with shape (809307665,) and data type float64

In [9]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal
from spreg import OLS, ML_Lag, ML_Error
from spreg.diagnostics import likratiotest
import time
from scipy.stats import chi2

# 文件夹路径
grid_folder = r'E:\seoul\480_based'

# 初始化 DataFrame 保存结果
lr_results = pd.DataFrame(columns=[
    'Year', 'City', 'Dependent', 'Test', 'LR_stat', 'df', 'p_value'
])
model_info = pd.DataFrame(columns=[
    'Year', 'City', 'Dependent', 'Model', 'Variable', 'Coef', 'Std_err', 't_stat', 'p_value', 'R2', 'Adj_R2', 'Rho'
])

# 循环年份
for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV', 'NDVI', 'SVF', 'EV', 'WR',
                        'Dist_P', 'Dist_M', 'Dist_W']
    explanatory_vars_clean = ['BCR', 'BHV', 'NDVI', 'SVF', 'EV', 'WR']
    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp') and filename.startswith(f'city{year}'):
            path = os.path.join(grid_folder, filename)
            print(f"\nProcessing {filename}")
            gdf = gpd.read_file(path)

            # 构造空间权重矩阵（1000米阈值距离）
            threshold = 1000
            w = libpysal.weights.DistanceBand.from_dataframe(gdf, threshold=threshold, binary=False)
            w.transform = 'r'

            # 构造 y, X
            X0 = gdf[explanatory_vars].values
            x = np.hstack([np.ones((len(gdf), 1)), X0])  # 加常数列

            city_name = filename.split('_')[0]
            w_name = 'W1000'
            ds_name = f'yr{year}'
            slx_vars_bool = [v in explanatory_vars_clean for v in explanatory_vars]
            print(slx_vars_bool)
            for target in target_vars:
                yi = gdf[[target]].values
                models = {}

                # 1. OLS
                models['OLS'] = OLS(yi, x, w=w, spat_diag=True, moran=True,
                                    name_w=w_name, name_ds=ds_name)
                # SLX
                models['SLX'] = OLS(yi, x, w=w, slx_lags=1,slx_vars=slx_vars_bool, # type: ignore
                                     spat_diag=True, moran=True,name_w=w_name,name_ds=ds_name)
                
                # 2. SLM
                models['SLM'] = ML_Lag(yi, x, w=w, method="full",
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['full'])

                # 3. SDM
                models['SDM'] = ML_Lag(yi, x, w=w, slx_lags=1, slx_vars=slx_vars_bool, # type: ignore
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['full'],
                                       spat_diag=True)

                # 4. SEM
                models['SEM']  = ML_Error(yi, x, w=w, method="full",
                     name_w=w_name,name_ds=ds_name)

                # 5. SDEM
                models['SDEM']  = ML_Error(yi, x, w=w, slx_lags=1, method="full",slx_vars=slx_vars_bool, # type: ignore
                    name_w=w_name, name_ds=ds_name)


                # 保存模型系数信息
                for mdl_name, mdl in models.items():
                    coef = mdl.betas.flatten()
                    var_names = ['const'] + explanatory_vars
                    if mdl_name in ['SLM', 'SDM']:
                        var_names += [f"W_{v}" for v in explanatory_vars]
                    if mdl_name == 'SDEM':
                        var_names += [f"W_{v}" for v in explanatory_vars]  # SDEM Durbin
                    se = mdl.std_err.flatten()
                    t_stat = coef / se
                    p_val = 2 * (1 - chi2.cdf(t_stat**2, 1))
                    R2 = getattr(mdl, 'r2', np.nan)
                    R2_adj = getattr(mdl, 'ar2', np.nan)
                    Rho = getattr(mdl, 'rho', np.nan)
                    if mdl_name in ['SEM', 'SDEM']:
                        Rho = getattr(mdl, 'lambda', np.nan)

                    for c, v, s, t, p in zip(coef, var_names, se, t_stat, p_val):
                        model_info = pd.concat([model_info, pd.DataFrame([{
                            'Year': year,
                            'City': city_name,
                            'Dependent': target,
                            'Model': mdl_name,
                            'Variable': v,
                            'Coef': c,
                            'Std_err': s,
                            't_stat': t,
                            'p_value': p,
                            'R2': R2,
                            'Adj_R2': R2_adj,
                            'Rho': Rho
                        }])], ignore_index=True)
                def safe_likratiotest(m0, m1):
                    res = likratiotest(m0, m1)
                    df_expected = len(m1.betas.flatten()) - len(m0.betas.flatten())
                    if res['df'] != df_expected:
                        print(f"⚠️ df mismatch: likratiotest={res['df']}, expected={df_expected}")
                        res['df'] = df_expected
                    return res
                # 提取 LR 检验
                print(models['SDM'].summary)
                coefs = models['SDM'].betas.flatten()
                vars_ = models['SDM'].name_x
                print(models['SDM'].rho)
                print(coefs)
                print(vars_)
                lr_tests = {
                    'SLX-OLS': safe_likratiotest(models['OLS'], models['SLX']),
                    'SLM-OLS': safe_likratiotest(models['OLS'], models['SLM']),
                    'SDM-OLS': safe_likratiotest(models['OLS'], models['SDM']),
                    'SEM-OLS': safe_likratiotest(models['OLS'], models['SEM']),
                    'SDM-SLX': safe_likratiotest(models['SLX'], models['SDM']),
                    'SDM-SLM': safe_likratiotest(models['SLM'], models['SDM']),
                    'SDM-SEM': safe_likratiotest(models['SEM'], models['SDM']),
                    'SDEM-SLX': safe_likratiotest(models['SLX'], models['SDEM']),
                    'SDEM-SEM': safe_likratiotest(models['SEM'], models['SDEM']),
                }

                for test_name, res in lr_tests.items():
                    lr_results = pd.concat([lr_results, pd.DataFrame([{
                        'Year': year,
                        'City': city_name,
                        'Dependent': target,
                        'Test': test_name,
                        'LR_stat': res['likr'],
                        'df': res['df'],
                        'p_value': res['p-value']
                    }])], ignore_index=True)
                    print(f"{test_name}: df(likratiotest)={res['df']}, "
                            f"params_m0={len(models[test_name.split('-')[0]].betas.flatten())}, "
                            f"params_m1={len(models[test_name.split('-')[1]].betas.flatten())}")



# 保存 Excel
output_path = r'E:\seoul\480_based\statistics\LR_test_results_partial.xlsx'
with pd.ExcelWriter(output_path) as writer:
    lr_results.to_excel(writer, sheet_name='LR_tests', index=False)
    model_info.to_excel(writer, sheet_name='Model_info', index=False)


print(f"All LR test results and model info saved to {output_path}")


Processing city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp
[True, True, True, True, True, True, False, False, False]


c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\sparse\_data.py:117: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\JUJUBE\anaconda3\lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\JUJUBE\anaconda3\lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(
c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  warn("Method 'bounded' does not support relative tolerance in x; "
c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute toleran

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG WITH SLX - SPATIAL DURBIN MODEL (METHOD = FULL)
-------------------------------------------------------------------------------------------------
Data set            :      yr2016
Weights matrix      :       W1000
Dependent Variable  :     dep_var                Number of Observations:        1855
Mean dependent var  :     31.0401                Number of Variables   :          17
S.D. dependent var  :      2.5076                Degrees of Freedom    :        1838
Pseudo R-squared    :      0.9292
Spatial Pseudo R-squared:  0.8610
Log likelihood      :  -1991.1925
Sigma-square ML     :      0.4462                Akaike info criterion :    4016.385
S.E of regression   :      0.6680                Schwarz criterion     :    4110.321

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability

c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  warn("Method 'bounded' does not support relative tolerance in x; "
c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  warn("Method 'bounded' does not support relative tolerance in x; "


REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG WITH SLX - SPATIAL DURBIN MODEL (METHOD = FULL)
-------------------------------------------------------------------------------------------------
Data set            :      yr2016
Weights matrix      :       W1000
Dependent Variable  :     dep_var                Number of Observations:        1855
Mean dependent var  :     41.5797                Number of Variables   :          17
S.D. dependent var  :      4.0420                Degrees of Freedom    :        1838
Pseudo R-squared    :      0.9573
Spatial Pseudo R-squared:  0.9075
Log likelihood      :  -2416.2572
Sigma-square ML     :      0.6981                Akaike info criterion :    4866.514
S.E of regression   :      0.8355                Schwarz criterion     :    4960.450

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability

c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  warn("Method 'bounded' does not support relative tolerance in x; "
c:\Users\JUJUBE\anaconda3\lib\site-packages\scipy\optimize\_minimize.py:892: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  warn("Method 'bounded' does not support relative tolerance in x; "


REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG WITH SLX - SPATIAL DURBIN MODEL (METHOD = FULL)
-------------------------------------------------------------------------------------------------
Data set            :      yr2016
Weights matrix      :       W1000
Dependent Variable  :     dep_var                Number of Observations:        1855
Mean dependent var  :    -10.5434                Number of Variables   :          17
S.D. dependent var  :      1.7817                Degrees of Freedom    :        1838
Pseudo R-squared    :      0.9482
Spatial Pseudo R-squared:  0.8263
Log likelihood      :  -1120.4186
Sigma-square ML     :      0.1652                Akaike info criterion :    2274.837
S.E of regression   :      0.4064                Schwarz criterion     :    2368.773

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability